# SARSA for Cliff-Walking Problem :-

### SARSA (State–Action–Reward–State–Action) is an on-policy reinforcement learning algorithm that updates its Q-values based on the actual actions taken by the agent, making it distinct from off-policy methods like Q-learning. It is particularly useful when you want the agent’s learning process to reflect its real behavior rather than hypothetical optimal actions.

### On-policy learning :-
### The agent learns from the actions it actually executes, not from the best possible action.


***
## Gymnasium is the modern, actively maintained fork of OpenAI’s Gym, providing a standardized API and a wide collection of reinforcement learning environments. 

In [23]:
!pip install gymnasium

In [24]:
!pip install "gymnasium[toy-text]"

***
### Gymnasium’s “toy text” environments are a collection of simple, lightweight reinforcement learning problems designed for quick experimentation, debugging, and teaching. They’re perfect for understanding algorithms like SARSA, Q-learning, and Monte Carlo methods before scaling up to more complex environments.

### In Python, import random gives you access to the random module, which provides functions for generating pseudo-random numbers and making random selections. It’s widely used in reinforcement learning (e.g., ε-greedy exploration), simulations, and toy text environments like Gymnasium’s FrozenLake or Taxi.


In [25]:
import gymnasium as gym
import numpy as np
import random

In [26]:
# sample env

# it will create the cliff-walking environment 
env = gym.make("CliffWalking-v1")


In [27]:
env

<OrderEnforcing<PassiveEnvChecker<CliffWalkingEnv<CliffWalking-v1>>>>

In [28]:
print(env.observation_space.n)  # states = 48 as here we are using 4 X 12 grid
print(env.action_space.n)  # actions

starting_state, _ = env.reset()   # 36 = 3 * 12 + 0

print(starting_state) # 36


48
4
36


***
# SARSA Implementation :-

## Here we are working on 'Cliff walking Problem'
## And in this problem, as soon as agent encounters cliff cell, it returns back to the starting state actually.

### In SARSA, the parameter γ (gamma) is the discount factor. It controls how much the agent values future rewards compared to immediate ones.

### Role of Episodes :-
- Episodes = independent runs of the environment until termination (goal reached, failure, or max steps).
- More episodes → more experience → better convergence of Q-values.
- Too few episodes → agent may not explore enough, leading to unstable or incomplete learning.

## In SARSA, the parameter ε (epsilon) is used in the ε-greedy exploration strategy. It controls how often the agent explores random actions versus exploiting the best-known action from its Q-table.


In [29]:
# Here we need to define some of these parameters firstly
gamma = 0.99  # gamma means the discount factor
alpha = 0.5  # learning rate
episodes = 500 # here we can also take episodes to be 1000 or 2000 at first
epsilon = 0.1 # means 10% exploration & 90% exploitation in epsilon-greedy policy 


### np.zeros((48, 4)) → Creates a 2D NumPy array filled with zeros.
### Shape (48, 4):
- 48 → Number of states in CliffWalking (the grid is 4 rows × 12 columns = 48 positions).
- 4 → Number of possible actions (up, right, down, left).
### Meaning :- Each row corresponds to a state, and each column corresponds to an action.
## Initially, all Q-values are set to 0 because the agent has no knowledge yet.


In [30]:
# We need this Q-table to find the optimal action :-

# Q-table => stores Q-values

Q = np.zeros((48, 4))

Q 

array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],


### Policy --> epsilon-greedy policy
### THis policy will tell what should be the action based on state

### env.action_space → This defines the set of all possible actions the agent can take in the environment.
- For example, in CartPole, the action space is {0, 1} (push left or right).
- In CliffWalking, the action space is {0, 1, 2, 3} (up, right, down, left).
### .sample() → This method randomly selects one valid action from the action space.
### return env.action_space.sample() → Returns a random action to be executed by the agent.

***
- Q[state] → This gives you the row of the Q-table corresponding to the current state.
Example: In CliffWalking, if state = 37, then Q[37] is a vector of 4 values (one for each action: up, right, down, left).
- np.argmax(Q[state]) → Finds the index of the maximum Q-value in that row.
- If Q[37] = [0.1, 0.5, -0.2, 0.3]
- Then np.argmax(Q[37]) = 1 (because 0.5 is the largest value, corresponding to action 1).
- return np.argmax(Q[state]) → Returns the action with the highest estimated value for the current state.
This is the greedy choice (exploitation).


In [31]:
# Policy --> epsilon-greedy policy : state -> action

def epsilon_greedy(state):
    if random.random() < epsilon:  # exploration case 
        return env.action_space.sample()  # return some random action ==> means exploration 
    else:
        return np.argmax(Q[state]) # exploitation step in an ε-greedy policy for SARSA
        

### What env.step(action) Does :-
- Takes an action: You pass in the action chosen by your policy (e.g., from np.argmax(Q[state]) or env.action_space.sample()).
- Moves the environment forward: The environment simulates what happens when that action is applied.

### - Returns a tuple of results:
- next_state → The new state after the action.
Example: In CliffWalking, moving right from state 37 might take you to state 38.
- reward → The immediate reward received.
- CliffWalking: usually -1 per step, -100 if you fall off the cliff, +0 if you reach the goal.
- terminated → Boolean flag indicating if the episode ended because the agent reached a terminal state (goal or failure).
- truncated → Boolean flag indicating if the episode ended because of a time limit or external cutoff.
- info → Extra diagnostic information (often unused, but can contain helpful debugging data).



***
### Calling env.close() at the end of your reinforcement learning experiment is important because it properly shuts down the environment and releases resources. 

## Why env.close() is Needed :-
### Resource cleanup
- Many environments (especially those with rendering windows or physics engines) allocate memory, graphics contexts, or background processes.
- env.close() ensures these are freed, preventing memory leaks.
### Graphics handling in JupyterLab
- If you use env.render(), a window or text output may remain open.
- Without env.close(), JupyterLab can hang or leave ghost processes running.
### Good practice in loops
- When running multiple environments (e.g., benchmarking SARSA vs Q-learning), failing to close them can accumulate unused resources.
- This slows down your notebook or even crashes the kernel.
### Consistency across libraries
- Gymnasium, Gym, and other RL libraries expect environments to be closed after use.
- It’s part of the API contract, similar to closing files after reading/writing


In [32]:
# SARSA Algorithm :-
## In this, we will train our agent for multiple episodes 

# Here we firstly need to create the env. for each episodes
for episode in range(episodes):

    env = gym.make("CliffWalking-v1")

    done = False  # it will tell when the episode has ended
    state, _ = env.reset()  # it will give curr state & some additional info which we don't need actually 
    action = epsilon_greedy(state)  # for any state, action will be decided by epsilon-greedy policy 

    total_reward = 0 # total reward in one episode
    episode_length = 0 # total moves in episode

    while not done:
        next_state, reward, terminated, truncated, _ = env.step(action)  # Here now agent will take or execute this action in the env & then will return 5 parameters 
        done = terminated or truncated
        next_action = epsilon_greedy(next_state)  # for this next state, this will return the next action as per the epsilon-greedy policy 

        # SARSA update :- in this we update the Q-table
        Q[state, action] += alpha * (reward + gamma*(Q[next_state, next_action] - Q[state, action]))

        state = next_state
        action = next_action

        total_reward += reward
        episode_length += 1

    print(f"episode = {episode+1}/500 : total reward = {total_reward} & episode length = {episode_length}")
    env.close()


episode = 1/500 : total reward = -76 & episode length = 76
episode = 2/500 : total reward = -2331 & episode length = 648
episode = 3/500 : total reward = -185 & episode length = 86
episode = 4/500 : total reward = -170 & episode length = 71
episode = 5/500 : total reward = -81 & episode length = 81
episode = 6/500 : total reward = -58 & episode length = 58
episode = 7/500 : total reward = -73 & episode length = 73
episode = 8/500 : total reward = -58 & episode length = 58
episode = 9/500 : total reward = -57 & episode length = 57
episode = 10/500 : total reward = -114 & episode length = 114
episode = 11/500 : total reward = -46 & episode length = 46
episode = 12/500 : total reward = -83 & episode length = 83
episode = 13/500 : total reward = -189 & episode length = 90
episode = 14/500 : total reward = -154 & episode length = 55
episode = 15/500 : total reward = -46 & episode length = 46
episode = 16/500 : total reward = -43 & episode length = 43
episode = 17/500 : total reward = -67 & 

***
## What did actually our agent learn? :-

### render_mode="human" :-
- Tells Gymnasium how to display the environment.
- "human" → Opens a visual window (or prints text for toy text environments).
- Other modes include "ansi" (text output for Jupyter), "rgb_array" (returns pixel arrays for custom rendering).


In [33]:
# Here now we want to see the env as humans 

env = gym.make("CliffWalking-v1", render_mode="human")
state, _ = env.reset()
done = False
total_reward = 0
episode_len = 0

while not done:
    # As now agent has already learned, so now no need to choose between exploit & explore
    # We now we can directly select the best optimal action using Q-table
    action = np.argmax(Q[state])
    # next_state, reward, terminated, truncated, _ = env.step(action)
    state, reward, terminated, truncated, _ = env.step(action)
    # Now no need to do :- next_state = state
    done = terminated or truncated 

    # next_state = state   # here we either need to do this or we can simply take the name for next_state return bu env.step() as state only
    
    # Now also no need to do SARSA update as that is only done in training process & that is already been done
    total_reward += reward
    episode_len += 1

print(f"total reward = {total_reward} & episode length = {episode_len}")
env.close()


total reward = -17 & episode length = 17


## Here this SARSA algorithm giving -17 which indicates the saffest shortest path. But not the exact shortest path as exact shortest path may not be safest.